# Train the priority classifier (v2.0) on Colab

DistilBERT (`distilbert-base-uncased`) fine-tuned on `insanar/prior-mail-priority` (config `v2`).

**Before you start**
- Runtime → Change runtime type → **T4 GPU**.
- This notebook clones the repo from GitHub; the migration lives on `main`.

## 1. Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime → T4 GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

## 2. Clone the repo

Private org repo → use a GitHub PAT with `repo` scope.

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub PAT (repo scope): ")
!git clone https://{TOKEN}@github.com/PJK-GM095-PIJAK/prior-mail-model.git
%cd prior-mail-model
# !git checkout <branch-or-sha>   # optional: pin a specific commit

## 3. Install dependencies

Not `pip install -e .` — Colab's Python doesn't satisfy the `>=3.11,<3.12` pin. Install the libs directly and put the repo root on `PYTHONPATH`. (`sentencepiece` is no longer needed — DistilBERT uses WordPiece.)

In [ ]:
!pip install -q transformers datasets accelerate evaluate wandb emoji huggingface_hub
%env PYTHONPATH=.

## 4. Experiment tracking (pick ONE)

Live tracking with W&B (project `priormail`):

In [ ]:
import wandb
wandb.login()   # paste your W&B key

...or skip tracking. If so, run **only** the line below in its own cell — with **no trailing comment** (a comment silently breaks `%env`):

In [ ]:
%env WANDB_MODE=offline

## 5. Build the dataset

Downloads `insanar/prior-mail-priority:v2` (public) and writes `data/processed/priority`.

In [ ]:
!make data

## 6. Train

DistilBERT on ~4.3k examples × 4 epochs ≈ a few minutes on a T4. The config requests `bf16`; on a T4 (no bf16) the trainer auto-falls back to `fp16`. Class weights (urgent/high ≈ 3.4×) apply automatically via `class_weights: balanced`.

In [ ]:
!make train config=configs/priority_v2.yaml

## 7. Evaluate against the gates

Promotion gates: macro-F1 ≥ 0.80, per-class recall ≥ 0.65, p95 < 500 ms on CPU.

In [ ]:
!make eval config=configs/priority_v2.yaml

## 8. Save the checkpoint

Zip + download (Drive is flaky):

In [ ]:
!cd checkpoints && zip -qr priority_v2.zip priority_v2
from google.colab import files
files.download("checkpoints/priority_v2.zip")

Or publish to the HuggingFace Hub (needs a **write** token; the uploader handles the Xet stall itself):

In [ ]:
from huggingface_hub import login
login()   # HF token with write access
!python -m src.exporter.export --checkpoint checkpoints/priority_v2 --hf-org insanar --version v2.0